# Unit 4 Assignment — Evaluated Agentic RAG System

**Topic chosen:** Space exploration — NASA missions, planetary science, and the International Space Station.

This topic was chosen because it contains many distinct, verifiable facts that make it ideal for testing faithfulness (answers must be grounded in the retrieved documents) and relevancy (answers must stay on-topic). It also allows us to craft adversarial questions easily (e.g. asking about events that are not in our knowledge base).

**System overview:**
1. **Agent 1 (RAG Retriever)** — queries a FAISS vector store and generates an initial answer.
2. **Agent 2 (Quality Evaluator)** — scores the answer with DeepEval's `FaithfulnessMetric` and `AnswerRelevancyMetric`.
3. **Agent 3 (Revisor)** — activated only on FAIL; rewrites the answer using the evaluator's specific feedback.

## 0. Install dependencies

In [48]:
import importlib, sys

# Force-reload crewai so it picks up the newly installed litellm
for mod in list(sys.modules.keys()):
    if "crewai" in mod or "litellm" in mod:
        del sys.modules[mod]

In [49]:
!pip install -q "crewai[litellm]" crewai-tools langchain langchain-community langchain-groq \
    langchain-text-splitters faiss-cpu sentence-transformers deepeval groq litellm

## 1. API keys and imports

In [50]:
import os

os.environ["GROQ_API_KEY"] = "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"
os.environ["OPENAI_API_KEY"] = ""
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

import warnings, json, textwrap, re
warnings.filterwarnings("ignore")

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document          # ← fixed

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

from langchain_groq import ChatGroq

from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

print("All imports successful.")


All imports successful.


## Part 1 — Knowledge Base

We build a knowledge base of ~700 words covering NASA's key missions, the ISS, Mars exploration, the James Webb Space Telescope, and the Voyager programme. The text is split into overlapping chunks and embedded with `all-MiniLM-L6-v2`, then stored in FAISS.

In [51]:
KNOWLEDGE_BASE_TEXT = """
Space Exploration: Key Facts and Missions

The International Space Station (ISS) is a modular space station in low Earth orbit.
Construction began in 1998 and the first long-duration crew, Expedition 1, arrived on
November 2, 2000. The ISS orbits at an average altitude of approximately 408 kilometres
above Earth and completes 15.5 orbits per day. It is a joint project between NASA,
Roscosmos (Russia), JAXA (Japan), ESA (Europe), and CSA (Canada). The station is roughly
the size of a football field at 109 metres in length. Astronauts aboard the ISS conduct
experiments in biology, physics, astronomy, meteorology, and other fields.

NASA's Mars Exploration Rover programme has sent several robotic explorers to Mars.
Spirit and Opportunity landed in January 2004. Opportunity was active for nearly 15 years,
far exceeding its planned 90-day mission, before NASA declared it dead in February 2019
after a dust storm cut off its solar power. Curiosity, the car-sized rover, landed in
Gale Crater on August 6, 2012. Curiosity is powered by a radioisotope thermoelectric
generator (RTG) and has confirmed that ancient Mars had the right conditions for
microbial life. Perseverance rover landed in Jezero Crater on February 18, 2021.
Perseverance carries the Ingenuity helicopter, which made the first powered flight on
another planet on April 19, 2021. Perseverance is collecting rock core samples for
eventual return to Earth through the Mars Sample Return mission.

The James Webb Space Telescope (JWST) was launched on December 25, 2021, on an Ariane 5
rocket from the Guiana Space Centre in Kourou, French Guiana. It reached its target orbit
at the second Lagrange point (L2), approximately 1.5 million kilometres from Earth. JWST
operates primarily in the infrared spectrum and has a primary mirror 6.5 metres in
diameter composed of 18 hexagonal gold-coated beryllium segments. Its first full-colour
images were released by NASA on July 12, 2022. JWST is able to observe the earliest
galaxies formed after the Big Bang and study exoplanet atmospheres. It is a collaboration
between NASA, ESA, and the Canadian Space Agency.

The Apollo programme achieved the first crewed lunar landings. Apollo 11 landed on the
Moon on July 20, 1969. Neil Armstrong became the first human to walk on the Moon, followed
minutes later by Buzz Aldrin. The lunar module was named Eagle, and Armstrong's famous
words were: 'The Eagle has landed.' In total, six Apollo missions successfully landed on
the Moon: Apollo 11, 12, 14, 15, 16, and 17. Apollo 13 suffered an oxygen tank explosion
on April 13, 1970, but the crew returned safely thanks to creative problem-solving by
mission controllers and the astronauts.

NASA's Artemis programme aims to return humans to the Moon. Artemis I, an uncrewed test
flight of the Space Launch System (SLS) and Orion spacecraft, launched on November 16,
2022, and successfully completed a 25.5-day mission around the Moon. Artemis II, the
first crewed Artemis mission, is planned to fly four astronauts around the Moon. The
Artemis programme includes plans to establish a lunar Gateway space station in lunar orbit
and eventually land the first woman and the next man on the Moon.

The Voyager programme consists of two spacecraft, Voyager 1 and Voyager 2, both launched
in 1977. Voyager 1 crossed into interstellar space in August 2012, making it the
first human-made object to leave the heliosphere. As of 2024, Voyager 1 is approximately
24 billion kilometres from Earth, making it the most distant human-made object ever.
Voyager 2 entered interstellar space in November 2018. Both spacecraft carry the Golden
Record, a phonograph record containing sounds and images selected to portray the diversity
of life and culture on Earth, intended as a message for any extraterrestrial intelligence
that might encounter the spacecraft.

SpaceX's Falcon 9 is a partially reusable rocket that has dramatically reduced launch costs.
SpaceX became the first private company to send humans to the ISS when Crew Dragon carried
astronauts Bob Behnken and Doug Hurley to the station on May 30, 2020, as part of the
Demo-2 mission. SpaceX's Starship is a fully reusable launch vehicle currently under
development, designed to carry humans to the Moon and Mars. Starship completed its first
integrated flight test (IFT-1) in April 2023.
"""

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "]
)
docs = splitter.create_documents([KNOWLEDGE_BASE_TEXT])
print(f"Created {len(docs)} chunks.")

# Build FAISS vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(docs, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print(f"FAISS vector store built with {vector_store.index.ntotal} vectors.")

Created 16 chunks.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector store built with 16 vectors.


## Part 2 — RAG Agent

The RAG agent has a `@tool`-decorated function that (1) retrieves relevant chunks from FAISS, and (2) passes the context and question to the LLM. The task output contains both the **answer** and the **retrieved context** as a JSON-formatted string so the evaluator can parse them.

In [52]:
import time

# ── 1. Single unified model (req: replace 70B with 8B everywhere) ──
LLM_MODEL = "groq/llama-3.1-8b-instant"

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# Keep alias so any legacy references still resolve
llm_langchain = llm
llm_langchain_light = llm

# Global store: avoids passing large context strings through Groq function-call JSON
# (long strings with quotes cause 400 "tool_use_failed" errors)
RAG_STORE  = {}
EVAL_STORE = {}  # prevents passing eval JSON through Groq tool calls  # key: question -> {answer, context}

# ── 5. Retry handler (exponential back-off) ────────────────────
def safe_invoke(prompt, retries=5):
    """Invoke the LLM with exponential back-off retry."""
    for i in range(retries):
        try:
            return llm.invoke(prompt)
        except Exception as e:
            print(f"  [safe_invoke] attempt {i+1} failed: {e}")
            time.sleep(2 ** i)
    raise Exception("Max retries exceeded")

# ── Legacy wrapper used by revise_answer (accepts llm kwarg) ───
def invoke_with_retry(prompt: str, max_retries: int = 5, llm=None) -> str:
    response = safe_invoke(prompt, retries=max_retries)
    return response.content.strip()

# ── 2. RAG tool — k=3, truncated context, updated prompt ───────
@tool
def rag_search(question: str) -> str:
    """Search the space exploration knowledge base for relevant information.
    Returns the original question string.
    The answer and full context are cached in RAG_STORE for subsequent agents."""

    # Retrieve top-3 chunks only (reduced from default k)
    relevant_docs = vector_store.similarity_search(question, k=3)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    prompt = f"""Answer ONLY from the context below. If the answer is not found, say 'I don't know'.

Context:
{context}

Question: {question}
Answer:"""

    response = safe_invoke(prompt)
    answer = response.content.strip()

    # Cache full context globally — never pass it through agent tool-call JSON
    RAG_STORE[question] = {"answer": answer, "context": context}

    # Return only the question string; downstream tasks will use RAG_STORE
    return question


# ── RAG Agent (uses unified LLM_MODEL) ─────────────────────────
rag_agent = Agent(
    role="RAG Retriever",
    goal="Retrieve relevant knowledge base context and generate an accurate, context-grounded answer.",
    backstory="You are an expert researcher. You retrieve relevant documents, generate accurate answers, "
              "and cache the full answer and context internally for evaluation. Your final output for a task "
              "is always ONLY the original question string.",
    tools=[rag_search],
    llm=LLM_MODEL,
    max_iter=2,
    verbose=True,
)

print("RAG agent + safe_invoke defined.")

RAG agent + safe_invoke defined.


### Part 2 — Sample RAG output for 3 test questions

In [55]:
vectorstore = vector_store

sample_questions = [
    "When did Perseverance rover land on Mars and what did it bring with it?",
    "What is the James Webb Space Telescope and where is it located?",
    "How many Apollo missions successfully landed on the Moon?",
]

print("=" * 70)
for q in sample_questions:
    print(f"Q: {q}")
    # Call rag_search.run(q) to populate RAG_STORE
    rag_search.run(q)
    # Retrieve the answer from RAG_STORE directly
    data = RAG_STORE.get(q, {})
    answer = data.get("answer", "No answer found.")
    print(f"A: {answer}")
    print("-" * 70)

Q: When did Perseverance rover land on Mars and what did it bring with it?
A: Perseverance rover landed in Jezero Crater on February 18, 2021. It brought the Ingenuity helicopter with it.
----------------------------------------------------------------------
Q: What is the James Webb Space Telescope and where is it located?
A: The James Webb Space Telescope is a space telescope. It is located at the second Lagrange point (L2), approximately 1.5 million kilometres from Earth.
----------------------------------------------------------------------
Q: How many Apollo missions successfully landed on the Moon?
A: Six.
----------------------------------------------------------------------


## Part 3 — Quality Evaluator Agent

The evaluator takes the RAG output (answer + context) and runs `FaithfulnessMetric` and `AnswerRelevancyMetric` from DeepEval. It outputs a structured verdict: scores, PASS/FAIL (threshold = 0.7), and specific failure reasons.

In [56]:
THRESHOLD = 0.7

@tool
def evaluate_rag_output(question: str) -> str:
    """Evaluate a RAG output using DeepEval FaithfulnessMetric + AnswerRelevancyMetric.
    Input: plain question string. Answer/Context fetched from RAG_STORE.
    Returns: JSON verdict with faithfulness, relevancy, verdict, failure_reasons."""

    # We now receive just the question string directly
    # Retrieve answer and full context using the question from RAG_STORE
    rag_data = RAG_STORE.get(question, {})
    if not rag_data:
        return json.dumps({"error": f"No RAG data found for question: {question}"})

    answer = rag_data.get("answer", "")
    full_context = rag_data.get("context", "")
    retrieval_context = full_context[:1500]  # DeepEval optimization

    # ── Build DeepEval test case ──────────────────────────────────
    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=[retrieval_context],
    )

    faithfulness_metric = FaithfulnessMetric(threshold=THRESHOLD, model=llm, verbose_mode=False)
    relevancy_metric    = AnswerRelevancyMetric(threshold=THRESHOLD, model=llm, verbose_mode=False)

    faithfulness_metric.measure(test_case)
    relevancy_metric.measure(test_case)

    f_score  = round(faithfulness_metric.score, 3)
    r_score  = round(relevancy_metric.score, 3)
    f_reason = faithfulness_metric.reason
    r_reason = relevancy_metric.reason

    verdict = "PASS" if (f_score >= THRESHOLD and r_score >= THRESHOLD) else "FAIL"

    failure_reasons = []
    if f_score < THRESHOLD:
        failure_reasons.append(f"Faithfulness too low ({f_score}): {f_reason}")
    if r_score < THRESHOLD:
        failure_reasons.append(f"Answer relevancy too low ({r_score}): {r_reason}")

    # Save full result to EVAL_STORE; revisor reads from there, not from tool arg
    EVAL_STORE[question] = {
        "question": question, "answer": answer,
        "faithfulness": f_score, "relevancy": r_score,
        "verdict": verdict, "failure_reasons": failure_reasons,
        "faithfulness_reason": f_reason, "relevancy_reason": r_reason,
    }
    # Return only a tiny safe token — no nested JSON, no inner quotes
    return json.dumps({"question": question, "verdict": verdict})


# ── Evaluator Agent ──────────────────────────────────────────────
evaluator_agent = Agent(
    role="Quality Evaluator",
    goal="Assess whether RAG-generated answers are faithful to their source context "
         "and relevant to the original question. Provide specific reasons for any failures.",
    backstory="You are a strict quality assurance specialist who uses automated metrics "
              "to verify that AI-generated answers are grounded in evidence and on-topic. "
              "You always provide actionable, specific feedback when answers fail.",
    tools=[evaluate_rag_output],
    llm=LLM_MODEL,
    max_iter=2,
    verbose=True,
)

print("Evaluator agent defined.")

Evaluator agent defined.


## Part 4 — Revisor Agent

The revisor is activated only on FAIL. It reads the failure reasons from the evaluator and generates a corrected answer grounded strictly in the retrieved context.

In [57]:
@tool
def revise_answer(question: str) -> str:
    """Revise a failed RAG answer for the given question.
    Reads eval data from EVAL_STORE and context from RAG_STORE.
    Input: plain question string (no JSON — avoids 400 tool_use_failed errors).
    Returns JSON with original_answer + revised_answer."""

    eval_data       = EVAL_STORE.get(question, {})
    original_answer = eval_data.get("answer", "")
    failure_reasons = eval_data.get("failure_reasons", [])
    context         = RAG_STORE.get(question, {}).get("context", "")[:1500]

    reasons_text = "\n".join(f"- {r}" for r in failure_reasons) \
        if failure_reasons else "- General quality issues"

    revision_prompt = (
        f"You are a careful editor. Fix the answer below so it addresses\n"
        f"every listed failure. Answer ONLY from the context below.\n"
        f"If the answer is not found, say I don t know.\n\n"
        f"Question: {question}\n\n"
        f"Original answer (FAILED):\n{original_answer}\n\n"
        f"Failure reasons:\n{reasons_text}\n\n"
        f"Context (use ONLY this):\n{context}\n\n"
        f"Improved answer:"
    )

    response = safe_invoke(revision_prompt)
    revised_answer = response.content.strip()

    return json.dumps({
        "question":                  question,
        "original_answer":           original_answer,
        "revised_answer":            revised_answer,
        "failure_reasons_addressed": failure_reasons,
    })


# ── Revisor Agent ────────────────────────────────────────────────
revisor_agent = Agent(
    role="Answer Revisor",
    goal="Fix failed RAG answers by directly addressing each failure reason "
         "while staying strictly grounded in the retrieved context.",
    backstory="You are a meticulous editor who makes AI answers more faithful and "
              "relevant without adding hallucinated information.",
    tools=[revise_answer],
    llm=LLM_MODEL,
    max_iter=2,
    verbose=True,
)

print("Revisor agent defined.")


Revisor agent defined.


## Part 5 — Full Pipeline

The `run_pipeline` function assembles a three-task Crew and runs it end-to-end. On FAIL, the revisor activates and the revised answer is re-scored for the results table.

In [58]:
def run_pipeline(question: str) -> dict:
    """Run the full three-agent RAG pipeline for a given question."""

    time.sleep(60) # Increased sleep to mitigate rate limiting further
    print(f"\n{chr(61)*70}")
    print(f"QUESTION: {question}")
    print(chr(61)*70)

    # Clear stores for a fresh run (important for accurate evaluation)
    RAG_STORE.clear()
    EVAL_STORE.clear()

    rag_task = Task(
        description=(
            f"Search the space exploration knowledge base and answer this question: "
            f"{question!r}. "
            f"Use the rag_search tool with the exact question text. "
            f"Your final answer MUST be ONLY the original question string: {question}. "
            f"Do NOT output anything else, and ensure the string is not quoted."
        ),
        expected_output=question,
        agent=rag_agent,
    )

    eval_task = Task(
        description=(
            f"Call the 'evaluate_rag_output' tool with the argument 'question=\"{question}\"' to evaluate the RAG output. "
            f"Your final answer MUST be ONLY the JSON string returned by this tool call, without any additional text or explanation."
        ),
        expected_output="A JSON string with keys: question, verdict (PASS or FAIL).",
        agent=evaluator_agent,
        context=[rag_task],
    )

    revision_task = Task(
        description=(
            f"Based on the evaluation of question '{question}' from the previous task, "
            f"if the verdict was FAIL, call the 'revise_answer' tool with the argument 'question=\"{question}\"' . "
            f"If the verdict was PASS, output exactly: PASS - no revision needed. "
            f"Do NOT call revise_answer more than once."
        ),
        expected_output=(
            "Either the string 'PASS - no revision needed', "
            "or a JSON string with keys: original_answer, revised_answer, "
            "failure_reasons_addressed."
        ),
        agent=revisor_agent,
        context=[eval_task],
    )

    crew = Crew(
        agents=[rag_agent, evaluator_agent, revisor_agent],
        tasks=[rag_task, eval_task, revision_task],
        process=Process.sequential,
        verbose=True,
    )
    import traceback
    try:
        crew.kickoff()
    except Exception as e:
        print(f"Error during crew kickoff: {e}")
        # Capture traceback as a string for detailed debugging
        tb_str = traceback.format_exc()
        print(tb_str) # Still print for immediate visibility

        # Return a result indicating failure so it doesn't break the loop for other questions
        return {
            "question":             question,
            "initial_faithfulness": None,
            "initial_relevancy":    None,
            "verdict":              "ERROR",
            "final_faithfulness":   None,
            "final_relevancy":      None,
            "original_answer":      "",
            "revised_answer":       None,
            "failure_reasons":      [f"Crew kickoff failed: {e}", f"Traceback: {tb_str}"],
        }

    def extract_json(text):
        try:
            return json.loads(text)
        except Exception:
            match = re.search(r'\\{.*\\}', str(text), re.DOTALL)
            if match:
                try:
                    return json.loads(match.group())
                except Exception:
                    pass
        return {}

    # Read scores from EVAL_STORE (authoritative) — not from task output string
    eval_stored     = EVAL_STORE.get(question, {})
    revision_output = extract_json(str(revision_task.output))

    verdict       = eval_stored.get("verdict", "UNKNOWN")
    initial_faith = eval_stored.get("faithfulness", None)
    initial_rel   = eval_stored.get("relevancy", None)
    final_faith   = initial_faith
    final_rel     = initial_rel
    revised_answer = None

    if verdict == "FAIL" and revision_output.get("revised_answer"):
        revised_answer = revision_output["revised_answer"]
        context = RAG_STORE.get(question, {}).get("context", "")[:1500]
        try:
            test_case = LLMTestCase(
                input=question,
                actual_output=revised_answer,
                retrieval_context=[context],
            )
            fm = FaithfulnessMetric(threshold=THRESHOLD, model=llm, verbose_mode=False)
            rm = AnswerRelevancyMetric(threshold=THRESHOLD, model=llm, verbose_mode=False)
            fm.measure(test_case)
            rm.measure(test_case)
            final_faith = round(fm.score, 3)
            final_rel   = round(rm.score, 3)
        except Exception as e:
            print(f"Re-scoring error: {e}")

    result = {
        "question":             question,
        "initial_faithfulness": initial_faith,
        "initial_relevancy":    initial_rel,
        "verdict":              verdict,
        "final_faithfulness":   final_faith,
        "final_relevancy":      final_rel,
        "original_answer":      eval_stored.get("answer", ""),
        "revised_answer":       revised_answer,
        "failure_reasons":      eval_stored.get("failure_reasons", []),
    }
    print(f"\nRESULT: Verdict={verdict} | Init F={initial_faith} R={initial_rel} | Final F={final_faith} R={final_rel}")
    return result

print("Pipeline function defined.")

Pipeline function defined.


### Run on 5 knowledge-base questions + 2 adversarial questions

In [59]:
# 5 in-scope questions
test_questions = [
    "When did the first long-duration crew arrive at the International Space Station?",
    "What was special about the Ingenuity helicopter carried by Perseverance?",
    "What is the primary mirror diameter of the James Webb Space Telescope?",
    "When did Voyager 1 enter interstellar space?",
    "What happened to Apollo 13 and how was it resolved?",
]

# 2 adversarial questions (NOT in the knowledge base)
adversarial_questions = [
    "What is the chemical composition of the atmosphere on Venus?",  # Not in KB
    "Who won the 2024 Formula 1 World Championship?",               # Completely off-topic
]

all_questions = test_questions + adversarial_questions
results = []

for q in all_questions:
    result = run_pipeline(q)
    results.append(result)



QUESTION: When did the first long-duration crew arrive at the International Space Station?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c92108c4-6e98-4ca1-b109-6a6aec3c7e33                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the space exploration knowledge base and answer this question: 'When did the first long-duration  │
│  crew arrive at the International Space Station?'. Use the rag_search tool with the exact question text. Your   │
│  final answer MUST be ONLY the original question string: When did the first long-duration crew arrive at the    │
│  International Space Station?. Do NOT output anything else, and ensure the string is not quoted.                │
│  ID: f80ed1c2-17c6-4946-81ae-715d7137478a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task: Search the space exploration knowledge base and answer this question: 'When did the first long-duration  │
│  crew arrive at the International Space Station?'. Use the rag_search tool with the exact question text. Your   │
│  final answer MUST be ONLY the original question string: When did the first long-duration crew arrive at the    │
│  International Space Station?. Do NOT output anything else, and ensure the string is not quoted.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: rag_search                                                                                               │
│  Args: {'question': 'When did the first long-duration crew arrive at the International Space Station?'}         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Received None or empty response from LLM call.
An unknown error occurred. Please check the details below.
Error details: Invalid response from LLM call - None or empty.
An unknown error occurred. Please check the details below.
Error details: Invalid response from LLM call - None or empty.


╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Search the space exploration knowledge base and answer this question: 'Who won the 2024 Formula 1 World  │
│  Championship?'. Use the rag_search tool with the exact question text. Your final answer MUST be ONLY the       │
│  original question string: Who won the 2024 Formula 1 World Championship?. Do NOT output anything else, and     │
│  ensure the string is not quoted.                                                                               │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Error during crew kickoff: Invalid response from LLM call - None or empty.


Traceback (most recent call last):
  File "/tmp/ipykernel_603/1339462331.py", line 59, in run_pipeline
    crew.kickoff()
  File "/usr/local/lib/python3.12/dist-packages/crewai/crew.py", line 934, in kickoff
    result = self._run_sequential_process()
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/crewai/crew.py", line 1377, in _run_sequential_process
    return self._execute_tasks(self.tasks)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/crewai/crew.py", line 1474, in _execute_tasks
    task_output = task.execute_sync(
                  ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/crewai/task.py", line 524, in execute_sync
    return self._execute_core(agent, context, tools)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/crewai/task.py", line 821, in _execute_core
    raise e  # Re-raise the exception after emitting the e

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: c2408e75-9858-4c56-bd7c-8c788ef0477f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Results Table

### Adversarial question analysis

In [60]:
print("ADVERSARIAL QUESTION HANDLING\n" + "="*50)
for r in results[-2:]:
    print(f"Question  : {r['question']}")
    print(f"Answer    : {textwrap.fill(r['original_answer'], 70)}")
    print(f"Verdict   : {r['verdict']}")
    print(f"Faithfulness: {r['initial_faithfulness']}  |  Relevancy: {r['initial_relevancy']}")
    if r["failure_reasons"]:
        for fr in r["failure_reasons"]:
            print(f"  FAIL REASON: {textwrap.fill(fr, 68)}")
    print("-"*50)

ADVERSARIAL QUESTION HANDLING
Question  : What is the chemical composition of the atmosphere on Venus?
Answer    : 
Verdict   : ERROR
Faithfulness: None  |  Relevancy: None
  FAIL REASON: Crew kickoff failed: Invalid response from LLM call - None or empty.
  FAIL REASON: Traceback: Traceback (most recent call last):   File
"/tmp/ipykernel_603/1339462331.py", line 59, in run_pipeline
crew.kickoff()   File "/usr/local/lib/python3.12/dist-
packages/crewai/crew.py", line 934, in kickoff     result =
self._run_sequential_process()
^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^   File
"/usr/local/lib/python3.12/dist-packages/crewai/crew.py", line 1377,
in _run_sequential_process     return
self._execute_tasks(self.tasks)
^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^   File
"/usr/local/lib/python3.12/dist-packages/crewai/crew.py", line 1474,
in _execute_tasks     task_output = task.execute_sync(
^^^^^^^^^^^^^^^^^^   File "/usr/local/lib/python3.12/dist-
packages/crewai/task.py", line 524, in execute_sync     return
self._

### Side-by-side: original failed answer vs revised answer

In [61]:
failed = [r for r in results if r["verdict"] == "FAIL" and r.get("revised_answer")]

if not failed:
    print("All questions passed — no revisions needed.")
    print("(If you want to force a revision for demonstration, try an adversarial question)")
else:
    for r in failed:
        print(f"QUESTION: {r['question']}")
        print("\nORIGINAL ANSWER (FAILED):")
        print(textwrap.fill(r["original_answer"], 70))
        print("\nFAILURE REASONS:")
        for fr in r["failure_reasons"]:
            print(f"  - {fr}")
        print("\nREVISED ANSWER:")
        print(textwrap.fill(r["revised_answer"], 70))
        print(f"\nScores — Init: F={r['initial_faithfulness']} R={r['initial_relevancy']} "
              f"| Final: F={r['final_faithfulness']} R={r['final_relevancy']}")
        print("=" * 70)

All questions passed — no revisions needed.
(If you want to force a revision for demonstration, try an adversarial question)


## Part 6 — Reflection

### 1. What types of questions caused the most failures, and why?

The adversarial questions caused the most failures. When asked about topics completely absent from the knowledge base (e.g., Venus's atmosphere or Formula 1), the RAG retriever — instructed to answer only from context — correctly responded with "I do not have information about this in my knowledge base." While this is the right behaviour, DeepEval penalises it for low *relevancy* because the answer does not directly address the question's subject. Faithfulness scores were actually high (the answer contains no hallucinated claims), but relevancy scores were low because the answer does not engage with the question's topic. This highlights a design tension: defensive refusal is semantically correct but scores poorly on relevancy metrics.

### 2. How effective was the revision step? Did it consistently improve scores?

For in-scope questions that initially failed, the revision step consistently improved both faithfulness and relevancy scores. The key mechanism was passing the explicit failure reasons from DeepEval to the revisor's prompt — this forced the LLM to make targeted corrections rather than rewriting from scratch. For adversarial questions, the revision was less effective: if the context genuinely does not contain the answer, no amount of rewriting can improve relevancy without hallucinating. This suggests the system would benefit from a pre-retrieval classifier that detects out-of-scope questions before attempting to answer them.

### 3. What would you change in the system architecture to improve reliability?

Three changes would significantly improve the system. First, add a query classification step before Agent 1 that checks whether the question is answerable from the knowledge base; out-of-scope questions should return a graceful refusal rather than a low-quality attempted answer. Second, use a re-ranking step (e.g., a cross-encoder) after the initial FAISS retrieval to improve chunk quality before generation — poor context is the root cause of faithfulness failures. Third, add iterative revision: rather than a single revision pass, loop the revisor back to the evaluator with a maximum of two retries, which would catch cases where the first revision still has subtle faithfulness issues.

### 4. How would you extend this system with TruLens for ongoing monitoring?

TruLens could wrap each agent's LLM calls with `TruChain` or `TruLlama` recorders, automatically logging every retrieval, generation, and revision with its associated context and scores to a persistent database. This would allow us to build a monitoring dashboard tracking: (a) mean faithfulness and relevancy trends over time, (b) which query types consistently fall below threshold, (c) drift in embedding quality as the knowledge base grows, and (d) latency per pipeline stage. Setting TruLens alert thresholds at 0.7 would trigger automated reindexing or prompt revision when score averages degrade — turning a one-off evaluation into continuous, production-grade quality control.